## Импорт библиотек

In [40]:
import fitz
import pandas as pd
import glob
import os
import re

## Смотрим количество pdf-файлов

In [48]:
pdf_folder = "PDF"
all_pdf = glob.glob(os.path.join(pdf_folder, "*.pdf"))
print(f"Найдено PDF: {len(all_pdf)}")

Найдено PDF: 10


## Собираем данные

In [53]:
def extract_metadata_and_clean_text(pdf_path):
    doc = fitz.open(pdf_path)
    # Читаем первые 3 страницы для метаданных
    text_meta = ""
    for i in range(min(3, len(doc))):
        text_meta += doc[i].get_text("text") + "\n"
    
    # Полный текст для очистки
    full_text_raw = ""
    for page in doc:
        full_text_raw += page.get_text("text") + "\n"
    doc.close()

    lines = text_meta.split('\n')
    lines = [l.strip() for l in lines if l.strip()]

    author = ""
    date = ""
    rating = ""
    company = ""

    #  Поиск автора и даты
    date_pattern = re.compile(r'(назад|\d+\s+[а-я]+\s+в\s+\d+:\d+|\d+\s+[а-я]+\s+\d{4})', re.IGNORECASE)
    for i, line in enumerate(lines):
        if date_pattern.search(line):
            date = line
            if i > 0:
                prev = lines[i-1]
                if (len(prev) < 80 and 
                    not re.search(r'\d+(?:\.\d+)?K', prev) and 
                    not re.search(r'https?://', prev) and
                    not re.search(r'Блог компании', prev)):
                    author = prev
            break
    if not date:
        for i, line in enumerate(lines):
            if ' в ' in line and re.search(r'\d+:\d+', line):
                date = line
                if i > 0 and len(lines[i-1]) < 80:
                    author = lines[i-1]
                break

    # Рейтинг
    for line in lines:
        match = re.search(r'\b(\d+(?:\.\d+)?K)\b', line)
        if match:
            rating = match.group(1)
            break

    # Компания
    for i, line in enumerate(lines):
        if 'Блог компании' in line:
            match = re.search(r'Блог компании\s*(.*?)(?:,|$)', line)
            if match:
                company = match.group(1).strip()
            if not company and i+1 < len(lines):
                company = lines[i+1].strip()
            company = re.sub(r'https?://\S+', '', company)
            company = company.strip(' ,')
            if 'habr.com' in company:
                company = ''
            break
    if not company:
        for line in lines:
            match = re.search(r'\b([A-Za-z0-9\-]+\.(com|ru))\b', line)
            if match and 'habr.com' not in match.group(1):
                company = match.group(1)
                break

    # Очистка основного текста
    all_lines = full_text_raw.split('\n')
    all_lines = [l.strip() for l in all_lines if l.strip()]

    # Определяем мусорные паттерны
    garbage_patterns = [
        r'Охват за \d+ дней',
        r'Рейтинг',
        r'Подписчики',
        r'Подписаться',
        r'Войти',
        r'Все потоки',
        r'Блог компании',
        r'Общий рейтинг',
        r'РЕКЛАМА'
    ]

    # Исключаем строки, которые содержат только цифры, знаки препинания.
    start_idx = 0
    for i, line in enumerate(all_lines):
        if len(line) < 20:
            continue
        # Проверяем на мусор
        if any(re.search(p, line) for p in garbage_patterns):
            continue
        if line and (line[0].isalpha()):
            if not re.match(r'^[\d\.,]+$', line):
                start_idx = i
                break

    if start_idx == 0:
        for i, line in enumerate(all_lines):
            if 'Подписаться' in line and i+1 < len(all_lines):
                start_idx = i+1
                break

    cleaned_text = ' '.join(all_lines[start_idx:])
    # Убираем повторяющиеся пробелы
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)

    return author, date, rating, company, cleaned_text

records = []
for pdf_path in all_pdf:
    filename = os.path.basename(pdf_path)
    print(f"Обработка: {filename}")
    try:
        author, date, rating, company, cleaned_text = extract_metadata_and_clean_text(pdf_path)
        records.append({
            'filename': filename,
            'author': author,
            'date': date,
            'rating': rating,
            'company': company,
            'text': cleaned_text
        })
        print(f"Автор: {author or '—'}\nДата: {date or '—'}\nРейтинг: {rating or '—'}\nКомпания: {company or '—'}\n")
    except Exception as e:
        print(f"  Ошибка: {e}")
        records.append({
            'filename': filename,
            'author': '',
            'date': '',
            'rating': '',
            'company': '',
            'text': ''
        })

df = pd.DataFrame(records)
df.to_csv("dataset_final_clean.csv", index=False, encoding='utf-8-sig')

Обработка: 10 систем управления проектами в 2026 году от российски.pdf
Автор: maiya_dom
Дата: 15 дек 2025 в 13:11
Рейтинг: 16K
Компания: ЛидерТаск

Обработка: 10 прорывных разработок 2026 года по версии MIT Technol.pdf
Автор: IBS_habrablog
Дата: 4 фев в 10:07
Рейтинг: 32K
Компания: IBS

Обработка: Yarn 2 — Устанавливаем и разбираемся _ Хабр.pdf
Автор: agafonovim
Дата: 28 апр 2021 в 22:31
Рейтинг: 98K
Компания: —

Обработка: Внутреннее устройство веб-сервера. Часть 1_ От syscalls.pdf
Автор: endlessAI
Дата: 23 фев в 00:16
Рейтинг: 9.5K
Компания: —

Обработка: Китайский робот научился теннису на 5 часах данных — и .pdf
Автор: runaway_llm
Дата: 21 мар в 01:39
Рейтинг: 7.3K
Компания: —

Обработка: Может ли парусная яхта идти быстрее ветра_ _ Хабр.pdf
Автор: intdif
Дата: 21 мар в 01:59
Рейтинг: 14K
Компания: barque.ru

Обработка: Принципы DOD в C++_ Часть 1. Оптимизация структур _ Хаб.pdf
Автор: Bibest
Дата: 22 фев в 15:15
Рейтинг: 11K
Компания: —

Обработка: Скрытые сигналы_ как модели учат

In [47]:
df = pd.read_csv("dataset_final_clean.csv", encoding='utf-8-sig')

for i, row in df.iterrows():
    print(f"Файл: {row['filename']}")
    print(f"Автор: {row['author']}")
    print(f"Дата: {row['date']}")
    print(f"Рейтинг: {row['rating']}")
    print(f"Компания: {row['company']}")
    print("Текст (первые 300 символов):")
    print(row['text'][:300].replace('\n', ' ') + "...")
    print()

Файл: 10 систем управления проектами в 2026 году от российски.pdf
Автор: maiya_dom
Дата: 15 дек 2025 в 13:11
Рейтинг: 16K
Компания: ЛидерТаск
Текст (первые 300 символов):
После 2022 года выбор системы управления проектами в России перестал быть вопросом цены или количества функций. На первый план вышли доступность, юридическая корректность и предсказуемость работы. Многие зарубежные сервисы требуют VPN, не принимают оплату, ограничивают функциональность или в любой м...

Файл: 10 прорывных разработок 2026 года по версии MIT Technol.pdf
Автор: IBS_habrablog
Дата: 4 фев в 10:07
Рейтинг: 32K
Компания: IBS
Текст (первые 300 символов):
IBS – технологический партнер лидеров экономики 85,23 Рейтинг 550 Подписчики Подписаться Автор оригинала: MIT Technology Review Каждый год журнал Массачусетского технологического института MIT Technology Review публикует список из десяти технологий, которые, по мнению редакции, определят ближайшее б...

Файл: Yarn 2 — Устанавливаем и разбираемся _ Хабр.pdf
Ав